# Compare Benchmark Runs
**Source:** Based on the example [compare-hypothesis.ipynb](https://github.com/Kotlin/kotlinx-benchmark/blob/master/examples/compare-hypothesis.ipynb) of the `kotlinx-benchmark` repository.

Grabs the last two runs and compares their score.

In [41]:
%use serialization, dataframe, kandy

In [42]:
@Serializable
public data class Benchmark(
    public val benchmark: String,
    public val mode: String,
    public val warmupIterations: Int,
    public val warmupTime: String,
    public val measurementIterations: Int,
    public val measurementTime: String,
    public val primaryMetric: PrimaryMetric,
    public val secondaryMetrics: Map<String, PrimaryMetric>,
    public val params: JsonObject? = null
)

@Serializable
public data class PrimaryMetric(
    public val score: Double,
    public val scoreError: Double,
    public val scoreConfidence: List<Double>,
    public val scorePercentiles: Map<String, Double>,
    public val scoreUnit: String,
    public val rawData: List<List<Double>>,
)

In [43]:
val nameLookupMap = mapOf(
    Pair("io.karpfen.features.BaseLineBenchmark.run", "Base-line"),
    Pair("io.karpfen.features.TogglePointOverheadBenchmark.run","Toggle-point overhead"),
    Pair("io.karpfen.features.runtime.TickByTickOverhead.idle", "Tick-by-tick overhead (idle)"),
    Pair("io.karpfen.features.runtime.BreakpointOverhead.idle", "Breakpoint overhead (idle)"),
    Pair("io.karpfen.features.runtime.EventReplayOverhead.idle", "Event replay overhead (idle)"),
    Pair("io.karpfen.features.runtime.EventReplayRecordingOverhead.record", "Event replay overhead (recording)"),
    Pair("io.karpfen.features.runtime.EventReplayInjectionOverhead.inject", "Event replay overhead (injection)"),
    Pair("io.karpfen.features.language.HistoryOverhead.idle", "History overhead (idle)"),
    Pair("io.karpfen.features.language.TickLimitOverhead.idle", "Tick limit overhead (idle)"),
    Pair("io.karpfen.features.language.TickLimitOverheadSpecializedNotSupported.notSupported", "Tick limit overhead (specialized) (not supported)"),
    Pair("io.karpfen.features.language.TickLimitOverheadSpecializedSupported.supported", "Tick limit overhead (specialized) (supported)"),
    Pair("io.karpfen.features.language.HistoryOverheadSpecializedNotSupported.notSupported", "History overhead (specialized) (not supported)"),
    Pair("io.karpfen.features.language.HistoryOverheadSpecializedSupported.supported", "History overhead (specialized) (supported)"),
)

In [44]:
val benchmarkCount = 1

In [45]:
import java.nio.file.Files
import java.nio.file.attribute.BasicFileAttributes
import kotlin.io.path.exists
import kotlin.io.path.forEachDirectoryEntry
import kotlin.io.path.isDirectory
import kotlin.io.path.listDirectoryEntries
import kotlin.io.path.readText

val runsDir = notebook.workingDir.resolve("../../../build/reports/benchmarks/main")
val outputFiles = runsDir.listDirectoryEntries()
    .filter { it.isDirectory() }
    .sortedByDescending { dir -> Files.readAttributes(dir, BasicFileAttributes::class.java).creationTime() }
    .subList(0, benchmarkCount)
    .map { it.resolve("benchmark.json") }
outputFiles

[C:\Users\fabio\Desktop\Programmieren\karpfenProject\karpfenRuntime\execution-engine\src\benchmark\kotlin\..\..\..\build\reports\benchmarks\main\2026-07-26T11.52.19.9647709\benchmark.json]

In [46]:
val json = Json { ignoreUnknownKeys = true }
val runs: List<List<Benchmark>> = outputFiles.map { file ->
    json.decodeFromString<List<Benchmark>>(file.readText())
}

In [47]:
import kotlinx.serialization.json.encodeToJsonElement
import org.jetbrains.letsPlot.core.plot.base.DataFrame

var combinedDf = emptyDataFrame<Benchmark>()
for (run in runs) {
    combinedDf = combinedDf.concat(run.toDataFrame() {
        "benchmark" from { nameLookupMap[it.benchmark] ?: it.benchmark }
        "score" from { it.primaryMetric.score }
        "scoreError" from { it.primaryMetric.scoreError }
        "deviationMin" from { it.primaryMetric.score - it.primaryMetric.scoreError }
        "deviationMax" from { it.primaryMetric.score + it.primaryMetric.scoreError }
    })
}
combinedDf = combinedDf.groupBy("benchmark").mean()
combinedDf

benchmark,score,scoreError,deviationMin,deviationMax
Toggle-point overhead,"8,480399","0,083615","8,396784","8,564014"
History overhead (idle),"8,670102","0,062697","8,607405","8,732799"
History overhead (specialized) (not s...,"6,092588","0,101260","5,991328","6,193848"
History overhead (specialized) (suppo...,"3,788612","0,058396","3,730216","3,847007"
Tick limit overhead (idle),"431,104449","1639,097819","-1207,993369","2070,202268"
Tick limit overhead (specialized) (no...,"7,297168","0,036538","7,260630","7,333706"
Tick limit overhead (specialized) (su...,"2,357402","0,009607","2,347796","2,367009"
Breakpoint overhead (idle),"9,145020","0,048554","9,096466","9,193573"
Event replay overhead (injection),"9,052296","0,037157","9,015139","9,089453"
Event replay overhead (idle),"16,467014","30,135963","-13,668950","46,602977"


In [48]:
val minScore = combinedDf["deviationMin"].values().map { (it as Number).toDouble() }.minOrNull() ?: 0.0
val maxScore = combinedDf["deviationMax"].values().map { (it as Number).toDouble() }.maxOrNull() ?: 0.0

val lowerBound = minScore * 0.99
val upperBound = maxScore * 1.01

val plot = combinedDf.sortBy {"score".desc()}.plot {
    bars {
        x("benchmark")
        y("score") {
            scale = continuous(lowerBound..upperBound)
        }
    }
    errorBars {
        x("benchmark")
        yMin("deviationMin")
        yMax("deviationMax")
        width = 0.2
        borderLine.color = Color.BLACK
        borderLine.width = 0.75
    }
    coordinatesTransformation = CoordinatesTransformation.cartesianFlipped()
    layout {
        this.xAxisLabel = ""
        this.yAxisLabel = "ms/1000 ticks"
        style {
            global {
                title {
                    margin(10.0, -10.0)
                }
                text {
                    fontFamily = FontFamily.MONO
                }
            }
        }
        // Adjust the height of the Kandy plot based on the number of tests.
        size = 700 to ((50 * combinedDf.size().nrow) + 100)
    }
}
DISPLAY(HTML("<h4>Comparison</h4>"))
DISPLAY(plot)

Comparison

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.8.2/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="4OV7e8" ></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 700.0, 
 height: 700.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("4OV7e8");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"mapping":{
},
"guides":{
"x":{
"title":""
},
"y":{
"title":"ms/1000 ticks"
}
},
"coord":{
"name":"flip",
"flip":true
},
"data":{
"score":[431.10444935722336,16.46701351517435,9.807432819859475,9.14501963114001,9.052295922238706,8.88072180940951,8.67010162209311,8.480398776393084,7.2971683349624685,6.092587763238849,3.788611579216345,2.357402267920316],
"deviationMax":[2070.202267941899,46.6029766143522,13.850326993430336,9.193573489187319,9.08945270202454,9.079056244596954,8.732798679430172,8.564013670123611,7.333706237485641,6.193847934370368,3.84700708427727,2.3670088708228816],
"deviationMin":[-1207.9933692274524,-13.668949584003503,5.764538646288614,9.0964657730927,9.015139142452872,8.682387374222065,8.607404564756049,8.396783882662557,7.260630432439296,5.99132759210733,3.73021607415542,2.3477956650177507],
"benchmark":["Tick limit overhead (idle)","Event replay overhead (idle)","Event replay overhead (recording)","Breakpoint overhead (idle)","Event replay overhead (injection)","Tick-by-tick overhead (idle)","History overhead (idle)","Toggle-point overhead","Tick limit overhead (specialized) (not supported)","History overhead (specialized) (not supported)","History overhead (specialized) (supported)","Tick limit overhead (specialized) (supported)"]
},
"ggsize":{
"width":700.0,
"height":700.0
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"discrete":true
},{
"aesthetic":"y",
"limits":[-1195.9134355351778,2090.904290621318]
},{
"aesthetic":"x",
"discrete":true
}],
"layers":[{
"mapping":{
"x":"benchmark",
"y":"score"
},
"stat":"identity",
"sampling":"none",
"inherit_aes":false,
"position":"dodge",
"geom":"bar",
"data":{
}
},{
"mapping":{
"x":"benchmark",
"ymin":"deviationMin",
"ymax":"deviationMax"
},
"stat":"identity",
"color":"#000000",
"size":0.75,
"sampling":"none",
"width":0.2,
"inherit_aes":false,
"position":"dodge",
"geom":"errorbar",
"data":{
}
}],
"theme":{
"text":{
"family":"mono",
"blank":false
},
"title":{
"margin":[10.0,-10.0,10.0,-10.0],
"blank":false
},
"axis_ontop":false,
"axis_ontop_y":false,
"axis_ontop_x":false
},
"data_meta":{
"series_annotations":[{
"type":"str",
"column":"benchmark"
},{
"type":"float",
"column":"score"
},{
"type":"float",
"column":"deviationMin"
},{
"type":"float",
"column":"deviationMax"
}]
},
"spec_id":"26"
};
 fig = LetsPlot.buildPlotFromProcessedSpecs(plotSpec, containerDiv, sizing);
 } else {
 fig.updateView({});
 }
 }
 
 const renderImmediately = 
 forceImmediateRender || (
 sizing.width_mode === 'FIXED' && 
 (sizing.height_mode === 'FIXED' || sizing.height_mode === 'SCALED')
 );
 
 if (renderImmediately) {
 renderPlot();
 }
 
 if (!renderImmediately || responsive) {
 // Set up observer for initial sizing or continuous monitoring
 var observer = new ResizeObserver(function(entries) {
 for (let entry of entries) {
 if (entry.contentBoxSize && 
 entry.contentBoxSize[0].inlineSize > 0) {
 if (!responsive && observer) {
 observer.disconnect();
 observer = null;
 }
 renderPlot();
 if (!responsive) {
 break;
 }
 }
 }
 });
 
 observer.observe(containerDiv);
 }
 
 // ----------

In [49]:
val baseline = combinedDf["score"].values().map { (it as Number).toDouble() }.minOrNull()?: 0.0

var percentageDiffDf = combinedDf.mapToFrame {
    "benchmark" from { it.benchmark }
    "percentageDiff" from { (it.score - baseline) / baseline * 100 + 100}
    "percentageDeviationMin" from { (it.score - it.scoreError - baseline) / baseline * 100 + 100}
    "percentageDeviationMax" from { (it.score + it.scoreError - baseline) / baseline * 100 + 100}
}

percentageDiffDf

benchmark,percentageDiff,percentageDeviationMin,percentageDeviationMax
Toggle-point overhead,"359,734904","356,187995","363,281812"
History overhead (idle),"367,782018","365,122435","370,441600"
History overhead (specialized) (not s...,"258,444978","254,149564","262,740391"
History overhead (specialized) (suppo...,"160,711289","158,234177","163,188402"
Tick limit overhead (idle),"18287,267100","-51242,564142","87817,098342"
Tick limit overhead (specialized) (no...,"309,542772","307,992850","311,092695"
Tick limit overhead (specialized) (su...,"100,000000","99,592492","100,407508"
Breakpoint overhead (idle),"387,927837","385,868203","389,987471"
Event replay overhead (injection),"383,994537","382,418362","385,570712"
Event replay overhead (idle),"698,523699","-579,831019","1976,878416"


In [50]:
val minScore = percentageDiffDf["percentageDeviationMin"].values().map { (it as Number).toDouble() }.minOrNull() ?: 0.0
val maxScore = percentageDiffDf["percentageDeviationMax"].values().map { (it as Number).toDouble() }.maxOrNull() ?: 0.0

val lowerBound = minScore * 0.99
val upperBound = maxScore * 1.01

val plot = percentageDiffDf.sortBy {"percentageDiff".desc()}.plot {
    bars {
        x("benchmark")
        y("percentageDiff") {
            scale = continuous(lowerBound..upperBound)
            axis.breaks(format = "{.1f}%")
        }
    }
    errorBars {
        x("benchmark")
        yMin("percentageDeviationMin")
        yMax("percentageDeviationMax")
        width = 0.2
        borderLine.color = Color.BLACK
        borderLine.width = 0.75
    }
    coordinatesTransformation = CoordinatesTransformation.cartesianFlipped()
    layout {
        this.xAxisLabel = ""
        this.yAxisLabel = "execution time increase"
        style {
            global {
                title {
                    margin(10.0, -20.0)
                }
                text {
                    fontFamily = FontFamily.MONO
                }
            }
        }
        // Adjust the height of the Kandy plot based on the number of tests.
        size = 800 to ((50 * combinedDf.size().nrow) + 100)
    }
}
DISPLAY(HTML("<h4>Comparison</h4>"))
DISPLAY(plot)

Comparison

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.8.2/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="WREFuH" ></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 800.0, 
 height: 700.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("WREFuH");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"mapping":{
},
"guides":{
"x":{
"title":""
},
"y":{
"title":"execution time increase"
}
},
"coord":{
"name":"flip",
"flip":true
},
"data":{
"percentageDiff":[18287.26710004995,698.5236987025314,416.0271224525259,387.9278371615245,383.9945369283359,376.7164361491864,367.78201752311935,359.7349036180589,309.54277232455445,258.44497759873997,160.71128931926546,100.0],
"percentageDeviationMax":[87817.09834224505,1976.8784160653677,587.5249711051224,389.9874711369403,385.57071169882227,385.1296984034224,370.44159998769265,363.28181179187226,311.09269458518787,262.7403909233759,163.1884017686584,100.40750800291035],
"percentageDeviationMin":[-51242.564142145144,-579.8310186603051,244.52927379992934,385.8682031861087,382.4183621578495,368.30317389495036,365.1224350585461,356.1879954442456,307.9928500639212,254.14956427410405,158.23417686987256,99.59249199708965],
"benchmark":["Tick limit overhead (idle)","Event replay overhead (idle)","Event replay overhead (recording)","Breakpoint overhead (idle)","Event replay overhead (injection)","Tick-by-tick overhead (idle)","History overhead (idle)","Toggle-point overhead","Tick limit overhead (specialized) (not supported)","History overhead (specialized) (not supported)","History overhead (specialized) (supported)","Tick limit overhead (specialized) (supported)"]
},
"ggsize":{
"width":800.0,
"height":700.0
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"discrete":true
},{
"aesthetic":"y",
"format":"{.1f}%",
"limits":[-50730.13850072369,88695.2693256675]
},{
"aesthetic":"x",
"discrete":true
}],
"layers":[{
"mapping":{
"x":"benchmark",
"y":"percentageDiff"
},
"stat":"identity",
"sampling":"none",
"inherit_aes":false,
"position":"dodge",
"geom":"bar",
"data":{
}
},{
"mapping":{
"x":"benchmark",
"ymin":"percentageDeviationMin",
"ymax":"percentageDeviationMax"
},
"stat":"identity",
"color":"#000000",
"size":0.75,
"sampling":"none",
"width":0.2,
"inherit_aes":false,
"position":"dodge",
"geom":"errorbar",
"data":{
}
}],
"theme":{
"text":{
"family":"mono",
"blank":false
},
"title":{
"margin":[10.0,-20.0,10.0,-20.0],
"blank":false
},
"axis_ontop":false,
"axis_ontop_y":false,
"axis_ontop_x":false
},
"data_meta":{
"series_annotations":[{
"type":"str",
"column":"benchmark"
},{
"type":"float",
"column":"percentageDiff"
},{
"type":"float",
"column":"percentageDeviationMin"
},{
"type":"float",
"column":"percentageDeviationMax"
}]
},
"spec_id":"29"
};
 fig = LetsPlot.buildPlotFromProcessedSpecs(plotSpec, containerDiv, sizing);
 } else {
 fig.updateView({});
 }
 }
 
 const renderImmediately = 
 forceImmediateRender || (
 sizing.width_mode === 'FIXED' && 
 (sizing.height_mode === 'FIXED' || sizing.height_mode === 'SCALED')
 );
 
 if (renderImmediately) {
 renderPlot();
 }
 
 if (!renderImmediately || responsive) {
 // Set up observer for initial sizing or continuous monitoring
 var observer = new ResizeObserver(function(entries) {
 for (let entry of entries) {
 if (entry.contentBoxSize && 
 entry.contentBoxSize[0].inlineSize > 0) {
 if (!responsive && observer) {
 observer.disconnect();
 observer = 